# 00 · Setup — environment check

Every module starts with `%run ../setup/00_setup`. This notebook resolves the
participant catalog and exports shared variables.

In [ ]:
import re

CATALOG_PREFIX = "training_dbx_ana_medi"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DEFAULT_SCHEMA = "default"
VOLUME_NAME = "datasets"

current_user = spark.sql("SELECT current_user()").first()[0]
user_slug = re.sub(r"[^a-zA-Z0-9]", "_", current_user.split("@")[0]).lower()
user_slug = re.sub(r"_+", "_", user_slug).strip("_") or "user"

if "trainer" in current_user.lower() or "trener" in current_user.lower():
    user_slug = "trainer"

CATALOG = f"{CATALOG_PREFIX}_{user_slug}"
BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
DATASET_PATH = f"/Volumes/{CATALOG}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"

print("User:", current_user)
print("Catalog:", CATALOG)

In [ ]:
catalogs = {r[0] for r in spark.sql("SHOW CATALOGS").collect()}
if CATALOG not in catalogs:
    raise Exception(f"Catalog {CATALOG} not found. Ask the trainer to run setup/00_pre_config.")

spark.sql(f"USE CATALOG {CATALOG}")

schemas = {r[0] for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()}
missing = [s for s in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA] if s not in schemas]
if missing:
    raise Exception(f"Missing schemas in {CATALOG}: {missing}")

print("[OK] Environment ready")
print("BRONZE:", BRONZE)
print("SILVER:", SILVER)
print("GOLD:", GOLD)
print("DATASET_PATH:", DATASET_PATH)